## Tutorial to DoseAnnotation class

In [ ]:
import pyspark
import re
import dxpy
import hail as hl
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
import sys
sys.path.append('../')
from prescriptions_processing import DoseAnnotation

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'filtered_prescriptions_v6.2.0.ht'

output_database = 'prescriptions_db'
output_tb = 'filtered_prescriptions_with_doses_v6.2.0.ht'

project_id = dxpy.PROJECT_CONTEXT_ID

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=project_id)['id']
input_prescriptions_ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')

In [ ]:
annotator = DoseAnnotation(input_prescriptions_ht, input_dir='../data/codes_lkps')
ht_annotated = annotator.annotate()

In [ ]:
n_no_dose = ht_annotated.aggregate(hl.agg.count_where(hl.is_missing(ht_annotated.doses.value)))
n_no_quantity = ht_annotated.aggregate(hl.agg.count_where(hl.is_missing(ht_annotated.quantity.value)))
all_ht_count = ht_annotated.count()

In [ ]:
print('Fraction of records with no dose:')
print(f"{(n_no_dose / all_ht_count) * 100:.4f}%")
print('Fraction of records with no quantity:')
print(f"{(n_no_quantity / all_ht_count) * 100:.4f}%")

In [ ]:
ht_with_info = ht_annotated.annotate(
    quantity_missing = hl.is_missing(ht_annotated.quantity.value),
    dose_missing = hl.is_missing(ht_annotated.doses.value)
)

In [ ]:
person_stats = ht_with_info.group_by(ht_with_info.eid).aggregate(
    n_total = hl.agg.count(),
    n_quantity_missing = hl.agg.count_where(ht_with_info.quantity_missing),
    n_dose_missing = hl.agg.count_where(ht_with_info.dose_missing)
)

person_stats = person_stats.annotate(
    percent_quantity_missing = person_stats.n_quantity_missing / person_stats.n_total,
    percent_dose_missing = person_stats.n_dose_missing / person_stats.n_total
)

In [ ]:
percent_missing_quantity_person = person_stats.aggregate(hl.agg.collect(person_stats.percent_quantity_missing))
percent_missing_dose_person = person_stats.aggregate(hl.agg.collect(person_stats.percent_dose_missing))

In [ ]:
plt.hist(percent_missing_quantity_person, bins=20)
plt.xlabel('Fraction of records missing quantity (per person)')
plt.ylabel('Number of unique persons')
plt.title('Histogram of missing quantity per person')
plt.grid(True)
plt.show()

plt.hist(percent_missing_dose_person, bins=20)
plt.xlabel('Fraction of records missing dose (per person)')
plt.ylabel('Number of unique persons')
plt.title('Histogram of missing dose per person')
plt.grid(True)
plt.show()

In [ ]:
all_missing = person_stats.filter(person_stats.percent_quantity_missing == 1.0)
n_recepts = all_missing.aggregate(hl.agg.collect(all_missing.n_total))

In [ ]:
print("People with all recors missing:")
print(f"Number of individuals: {len(n_recepts)}")
print(f"Average number of prescriptions per individual: {sum(n_recepts)/len(n_recepts) if n_recepts else 0:.2f}")

In [ ]:
person_stats = ht_with_info.group_by(ht_with_info.eid).aggregate(
    n_recepts = hl.agg.count()
)

n_recepts_list = person_stats.aggregate(hl.agg.collect(person_stats.n_recepts))

plt.hist(n_recepts_list, bins=100)
plt.xlabel('Number of prescriptions per person')
plt.ylabel('Number of unique persons')
plt.title('Histogram of prescriptions per person')
plt.grid(True)
plt.show()

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=project_id)['id']
output_url = f'dnax://{output_db_id}/{output_tb}'

%time ht_annotated.write(output_url, overwrite=True)